# JS08 — Tugas Lab 1: Agglomerative pada Data Nasabah Kartu Kredit

**Praktikum Pembelajaran Mesin — Politeknik Negeri Malang**

| | |
|---|---|
| Nama | TBD |
| NIM | TBD |
| Kelas | TBD |
| Tanggal | TBD |

**Instruksi**
1. Gunakan dataset `CC_Customer_Dataset.csv` (dataset Tugas Lab JS07).
2. Buat model klasterisasi dengan **Agglomerative Clustering** memakai **seluruh variabel yang dapat digunakan**.
3. Buat *scatterplot* berdasarkan dua variabel: `Avg_Credit_Limit` dan `Total_Credit_Cards` (modul menulis `Total_credit_card`), lalu tunjukkan bentuk cluster-nya.
4. Tentukan nilai *Silhouette Score* dan *DBI* dari model.

**Prasyarat:** file `CC_Customer_Dataset.csv` di direktori kerja, atau unggah saat diminta (cell Langkah 0).


In [ ]:
# Import library dasar (dipakai di seluruh notebook ini)
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.preprocessing import StandardScaler

# Tugas Lab 1 — Agglomerative pada Data Nasabah Kartu Kredit

**Instruksi**
1. Gunakan dataset `CC_Customer_Dataset.csv` (dataset Tugas Lab JS07).
2. Buat model klasterisasi dengan **Agglomerative Clustering** memakai **seluruh variabel yang dapat digunakan**.
3. Buat *scatterplot* berdasarkan dua variabel: `Avg_Credit_Limit` dan `Total_Credit_Cards` (modul menulis `Total_credit_card`), lalu tunjukkan bentuk cluster-nya.
4. Tentukan nilai *Silhouette Score* dan *DBI* dari model.

## Langkah 0 — Menyiapkan Dataset

In [ ]:
TUGAS_PATH = "CC_Customer_Dataset.csv"

from pathlib import Path

if Path(TUGAS_PATH).exists():
    print(f"Ditemukan: {TUGAS_PATH}")
else:
    try:  # khusus Google Colab
        from google.colab import files
        print("Unggah file CC_Customer_Dataset.csv:")
        files.upload()
    except ImportError:
        print(f"Letakkan {TUGAS_PATH} di direktori kerja notebook ini.")

## Langkah 1 — Load & Inspeksi Data

Perhatikan nama kolom hasil cetak di bawah; nama kolom bisa sedikit berbeda antar versi dataset.

In [ ]:
dfc = pd.read_csv(TUGAS_PATH)
print("Shape:", dfc.shape)
print("Kolom :", list(dfc.columns))
dfc.info()

In [ ]:
dfc.head()

In [ ]:
dfc.describe().T

## Langkah 2 — Scatterplot `Avg_Credit_Limit` vs `Total_Credit_Cards`

**Pertanyaan:** apakah sebaran data dapat dikatakan linier?

In [ ]:
import re

def cari_kolom(df, pola):
    """Cari nama kolom yang cocok dengan pola regex."""
    for kol in df.columns:
        if re.search(pola, str(kol), flags=re.IGNORECASE):
            return kol
    raise KeyError(f"Kolom dengan pola {pola!r} tidak ditemukan di {list(df.columns)}")

KOL_LIMIT = cari_kolom(dfc, r'credit_?limit')
KOL_CARD  = cari_kolom(dfc, r'credit_?cards?$')
print("Fitur scatter:", KOL_LIMIT, "|", KOL_CARD)

plt.figure(figsize=(7, 5))
plt.scatter(dfc[KOL_LIMIT], dfc[KOL_CARD], s=40, alpha=0.7, c='steelblue', edgecolor='k', linewidth=0.3)
plt.title(f'Sebaran {KOL_LIMIT} vs {KOL_CARD}')
plt.xlabel(KOL_LIMIT)
plt.ylabel(KOL_CARD)
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# [tambahan] Korelasi antar semua variabel numerik
dfc.corr(numeric_only=True).round(3)

**Analisis:** TBD

*(Isi: apakah sebaran membentuk garis lurus / ada pola tertentu, dan nilai korelasinya.)*

## Langkah 3 — Seleksi Fitur & Standarisasi

Seluruh variabel yang dapat digunakan = semua kolom numerik, **kecuali** kolom identitas (`Sl_No`, `Customer Key`). Standarisasi (`StandardScaler`) diperlukan karena `Avg_Credit_Limit` bersatuan ratusan ribu, sedangkan `Total_Credit_Cards` hanya satuan — tanpa scaling, jarak Euclidean akan didominasi `Avg_Credit_Limit`.

In [ ]:
from sklearn.preprocessing import StandardScaler

KOL_ID = r'(sl[\s_]?no|customer[\s_]?key|^id$|^no$)'
fitur = dfc.drop(columns=[k for k in dfc.columns
                          if re.search(KOL_ID, str(k), flags=re.IGNORECASE)
                          or not np.issubdtype(dfc[k].dtype, np.number)])
print("Fitur dipakai:", list(fitur.columns))

Xc_raw = fitur.values
Xc = StandardScaler().fit_transform(Xc_raw)
print("Shape matriks fitur:", Xc.shape)

## Langkah 4 — Model Agglomerative + Evaluasi

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score

# [tambahan] pilih jumlah cluster terbaik pada rentang k = 2..8
rows = []
for k in range(2, 9):
    y = AgglomerativeClustering(n_clusters=k, metric='euclidean', linkage='ward').fit_predict(Xc)
    rows.append({'n_clusters': k,
                 'silhouette': silhouette_score(Xc, y),
                 'dbi': davies_bouldin_score(Xc, y)})

hasil_tugas = pd.DataFrame(rows).set_index('n_clusters').round(4)
hasil_tugas

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(hasil_tugas.index, hasil_tugas['silhouette'], marker='o', color='tab:blue')
ax1.set_xlabel('Jumlah cluster (k)')
ax1.set_ylabel('Silhouette Score', color='tab:blue')
ax1.set_xticks(hasil_tugas.index)

ax2 = ax1.twinx()
ax2.plot(hasil_tugas.index, hasil_tugas['dbi'], marker='s', color='tab:red')
ax2.set_ylabel('Davies-Bouldin Index', color='tab:red')

plt.title('Pemilihan jumlah cluster — Agglomerative (data nasabah)')
plt.tight_layout()
plt.show()

In [ ]:
# Model akhir: k terbaik menurut Silhouette Score
K_TERBAIK = int(hasil_tugas['silhouette'].idxmax())
print("k terpilih:", K_TERBAIK)

hc_c = AgglomerativeClustering(n_clusters=K_TERBAIK, metric='euclidean', linkage='ward')
y_c = hc_c.fit_predict(Xc)

sil_c = silhouette_score(Xc, y_c)
dbi_c = davies_bouldin_score(Xc, y_c)
print(f"Silhouette Score     : {sil_c:.4f}")
print(f"Davies-Bouldin Index : {dbi_c:.4f}")
print("Jumlah anggota per cluster:")
print(pd.Series(y_c).value_counts().sort_index().to_string())

## Langkah 5 — Scatterplot Hasil Cluster

Cluster divisualisasikan pada dua fitur (`Avg_Credit_Limit` dan `Total_Credit_Cards`), tetapi model dilatih memakai **seluruh fitur**, sehingga cluster bisa tidak terpisah rapi pada bidang 2D ini.

In [ ]:
kolom_x = list(fitur.columns).index(KOL_LIMIT)
kolom_y = list(fitur.columns).index(KOL_CARD)

fig, ax = plt.subplots(figsize=(8, 6))
warna = plt.cm.tab10(np.linspace(0, 1, K_TERBAIK))

for c in range(K_TERBAIK):
    m = y_c == c
    ax.scatter(Xc_raw[m, kolom_x], Xc_raw[m, kolom_y], s=40, alpha=0.8,
               color=warna[c], edgecolor='k', linewidth=0.3,
               label=f'Cluster {c+1} (n={m.sum()})')

ax.set_title(f'Clusters of credit card customers (k={K_TERBAIK})')
ax.set_xlabel(KOL_LIMIT)
ax.set_ylabel(KOL_CARD)
ax.legend()
plt.tight_layout()
plt.show()

## Langkah 6 — Analisis

**Karakter tiap cluster:** TBD

*(Isi: rata-rata `Avg_Credit_Limit`, `Total_Credit_Cards`, `Total_visits_bank`, `Total_visits_online`, `Total_calls_made` per cluster, lalu jelaskan segmen nasabahnya.)*

In [ ]:
# [tambahan] Profil rata-rata tiap cluster (memudahkan penulisan analisis)
profil = fitur.copy()
profil['cluster'] = y_c
profil.groupby('cluster').mean().round(2)

---
## Kesimpulan Tugas Lab 1

**Apakah sebaran `Avg_Credit_Limit` vs `Total_Credit_Cards` linier?** TBD

**Jumlah cluster terpilih & alasannya:** TBD

**Silhouette Score / DBI model:** TBD

**Karakter tiap cluster (segmen nasabah):** TBD

*(Gunakan output tabel profil rata-rata per cluster di atas.)*